In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

In [2]:
df = pd.read_csv('yufeng_CTR_LoRA.csv')

In [3]:
df

,Unnamed: 0,test_id,headline,impressions,clicks,CTR
0,0,1,"Hey Dude. If You Have An Older Brother, There'...",4080.0,41.0,0.010049
1,1,1,"Here's The Science, Here's The Gay. Open Your ...",4069.0,54.0,0.013271
2,2,1,I've Got Some News For You. Being Gay Is Genet...,4160.0,40.0,0.009615
3,3,1,"SCIENCE FACT: Gay Science, Like Straight Scien...",4132.0,32.0,0.007744
4,4,1,If You Know Anyone Who Is Afraid Of Gay People...,4155.0,120.0,0.028881
...,...,...,...,...,...,...
77240,77240,17681,These Arab men get uncomfortable when asked to...,2000.0,31.0,0.015500
77241,77241,17681,There's a big taboo in Egypt that it's not oka...,2048.0,29.0,0.014160
77242,77242,17681,A cultural taboo in Egypt actually has people ...,2023.0,20.0,0.009886
77243,77243,17681,Moms aren't very visible in Egypt. Their sons ...,2036.0,28.0,0.013752


In [7]:
df = df[['test_id', 'headline', 'impressions', 'clicks', 'CTR']]
df.rename(columns={'new_test_id': 'test_id'}, inplace=True)



#For CTR LoRA, please use 70%(training) 10% 20% split, 10%+20% data is for LOLA
train_ratio = 0.7
calibrate_ratio = 0.1#to get 1,000 hyperparameter
test_ratio = 0.2


gss = GroupShuffleSplit(n_splits=1, test_size=test_ratio+calibrate_ratio, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['test_id']))
train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=test_ratio/(test_ratio+calibrate_ratio), random_state=42)
calibrate_idx, test_idx = next(gss2.split(temp_df, groups=temp_df['test_id']))
calibrate_df = temp_df.iloc[calibrate_idx]
test_df = temp_df.iloc[test_idx]
del temp_df
print(len(test_df))
print(len(calibrate_df))

unique_headlines_train = set(train_df['headline'].unique())
test_df = test_df[~test_df['headline'].isin(unique_headlines_train)]
calibrate_df = calibrate_df[~calibrate_df['headline'].isin(unique_headlines_train)]
print(len(test_df))
print(len(calibrate_df))

15483
7797
12039
6072
